In [ ]:
import numpy as np
from PIL import Image
from tqdm import tqdm
import pickle
import os
import sys

from skimage.segmentation import slic, felzenszwalb, quickshift
try:
    from skimage.segmentation import seeds
except ImportError:
    import skimage.segmentation as seg
    seeds = getattr(seg, 'seeds', None)
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

import matplotlib.pyplot as plt
from skimage.segmentation import mark_boundaries
from skimage.io import imread

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
if torch.cuda.is_available():
    try:
        torch.randn(1).to("cuda")
        device = torch.device("cuda")
    except:
        print("CUDA not supported on this GPU. Switching to CPU.")
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

# Feature Extractor

In [ ]:
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Use weights trained on the ImageNet dataset
        resnet = models.resnet152(pretrained=True)
        
        # Takes all layers except the last two (global average pooling and fully connected (classification)).
        self.features = nn.Sequential(*list(resnet.children())[:-2])  
        
        for p in self.features.parameters():
            p.requires_grad = False  # Freeze backbone

    def forward(self, x):
        with torch.no_grad():
            feat_map = self.features(x)  # [B, 2048, 7, 7]
            
        return feat_map

In [ ]:
class EfficientNetFeatureExtractor(nn.Module):
    def __init__(self, model_name='efficientnet_b3', pretrained=True):
        super().__init__()
        
        effnet = getattr(models, model_name)(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None)
        
        # Remove the classifier (last layer)
        self.features = nn.Sequential(*list(effnet.children())[:-1])
        
        # Freeze backbone
        for p in self.features.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            feat_map = self.features(x)  # [B, 1280, H', W']
        return feat_map

In [ ]:
class InceptionNetFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        
        inception = models.inception_v3(pretrained=pretrained, aux_logits=True)
        
        # Remove the last FC layer
        self.features = nn.Sequential(*list(inception.children())[:-1])
        
        # Freeze backbone
        for p in self.features.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            feat_map = self.features(x)  # [B, 2048, H', W']
        return feat_map

In [ ]:
class VGG19FeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        
        vgg = models.vgg19(pretrained=pretrained)
        
        # Remove classifier, take only convolutional features
        self.features = vgg.features  # [B, 512, H', W']
        
        # Freeze backbone
        for p in self.features.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            feat_map = self.features(x)  # [B, 512, H', W']
        return feat_map

# Image preprocessing

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Region-level feature aggregation

In [ ]:
def get_segments(img_np, method='seeds', n_segments=50):
    """
    Wrapper for different superpixel algorithms.
    """
    if method == 'slic':
        return slic(img_np, n_segments=n_segments, compactness=10, start_label=0)
    
    elif method == 'felzenszwalb':
        # Felzenszwalb doesn't take n_segments directly. 
        # 'scale' controls the number of clusters. Larger scale = fewer segments.
        return felzenszwalb(img_np, scale=100, sigma=0.5, min_size=50)
    
    elif method == 'seeds':
        if seeds is not None:
            return seeds(img_np, num_superpixels=n_segments, num_levels=4)
        else:
            return quickshift(img_np, kernel_size=3, max_dist=6, ratio=0.5)
    else:
        raise ValueError(f"Unknown method: {method}")

In [ ]:
######################### SLIC REGIONS WITHOUT ORIGINAL IMAGE #############################
def get_region_features(feat_map, segments):
    """
    Extract mean+max pooled features for each SLIC region.
    feat_map: torch.Tensor [C,H,W]
    segments: np.array [H,W] region IDs
    """
    n_regions = len(np.unique(segments))
    region_features = []

    # Go through each superpixel one by one and compute its representative deep features
    for region_id in np.unique(segments):
        
        # Boolean array of shape [H, W]
        mask = torch.tensor(segments == region_id)
        
        # Returns all pixels of the region
        region_pixels = feat_map[:, mask]  # [C, num_pixels]
        
        # Check if a region have no valid pixels.
        if region_pixels.size(1) == 0:
            continue
        
        mean_feat = region_pixels.mean(dim=1)
        max_feat = region_pixels.max(dim=1)[0]
        
        region_features.append(torch.cat([mean_feat, max_feat], dim=0))  # [2C]

    return torch.stack(region_features)  # [N_regions, 2C]

# Region feature extraction

In [ ]:
def extract_and_save_features_all_methods(extractor, img_dir, save_path, method='slic', n_segments=50):
    print(f"Starting feature extraction using {method}...")
    extractor = extractor.to(device).eval()
    data_dict = {}

    for label_name, label in [("benign", 0), ("malignant", 1)]:
        folder_path = os.path.join(img_dir, label_name)
        if not os.path.exists(folder_path): continue

        img_files = [f for f in os.listdir(folder_path) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
        
        for fname in tqdm(img_files, desc=f"Processing {label_name}"):
            try:
                img_path = os.path.join(folder_path, fname)
                image = Image.open(img_path).convert("RGB")
                image_resized = image.resize((224, 224))
                
                # CNN Extraction
                x = transform(image_resized).unsqueeze(0).to(device)
                feat_map = extractor(x).squeeze(0).to(device)
                
                feat_map_up = F.interpolate(feat_map.unsqueeze(0), size=(224, 224), mode='bicubic', align_corners=False).squeeze(0)

                # Segmentation Switching
                img_np = np.array(image_resized)
                segments = get_segments(img_np, method=method, n_segments=n_segments)

                # Map CNN features to regions
                region_features = get_region_features(feat_map_up, segments)

                data_dict[fname] = {
                    "filename": fname,
                    "region_features": region_features.cpu(),
                    "segments": segments,
                    "label": label,
                    "method": method # Track which method was used
                }

            except Exception as e:
                print(f"Failed {fname}: {e}")
                continue

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as f:
        pickle.dump(data_dict, f)

# Main Function

In [ ]:
print(os.getcwd())

In [ ]:
if __name__ == "__main__":
    img_dir = "../skin-datasets/PAD-Process" 
    seg_count = 50
    data_name = "pad"
    methods = ['seeds', 'felzenszwalb']
    
    extractor = ResNetFeatureExtractor().to(device).eval()

    for m in methods:
        save_path = f"./slic/resnet/{data_name}/{data_name}_{m}_features_{seg_count}.pkl"
        extract_and_save_features_all_methods(extractor, img_dir, save_path, method=m, n_segments=seg_count)

In [ ]:
pkl_path = "slic/resnet/isic/isic_felzenszwalb_features_50.pkl"

with open(pkl_path, "rb") as f:
    data_dict = pickle.load(f)

print(f"Total images in pkl: {len(data_dict)}")

In [ ]:
fname = list(data_dict.keys())[0]
content = data_dict[fname]

print(f"Filename: {content['filename']}")
print(f"Label: {content['label']}")
print(f"Region features shape: {content['region_features'].shape}")
print(f"Segments shape: {content['segments'].shape}")

In [ ]:
segments = content['segments']
num_segments = len(set(segments.flatten()))
print(f"Number of segments (nodes) in this image: {num_segments}")

In [ ]:
# Load one example
img_dir = "../skin-datasets/ISIC-Process" 
fname = list(data_dict.keys())[0]
content = data_dict[fname]

print(f"Visualizing: {content['filename']}, Label: {content['label']}")

# Load original image
label_folder = "benign" if content["label"] == 0 else "malignant"
img_path = os.path.join(img_dir, label_folder, content["filename"])
image = imread(img_path)

# Resize to match the SLIC segmentation map
image_resized = resize(image, (224, 224), preserve_range=True, anti_aliasing=True).astype(np.uint8)

# Get segmentation map
segments = content["segments"]

# Overlay segmentation boundaries on the image
segmented_image = mark_boundaries(image_resized, segments, color=(0, 0, 1))

# Display
plt.figure(figsize=(4,4))
plt.imshow(segmented_image)
plt.axis("off")
plt.title(f"SLIC Segmentation")   #({len(np.unique(segments))} regions)
plt.show()

In [ ]:
image = imread(img_path)
image_resized = resize(image, (224, 224), preserve_range=True, anti_aliasing=True).astype(np.uint8)

plt.figure(figsize=(8, 8))

plt.subplot(1, 2, 1)
plt.imshow(image_resized)
plt.axis('off')
plt.title("Resized Image (224×224)")

plt.subplot(1, 2, 2)
plt.imshow(segments, cmap='nipy_spectral')
plt.axis('off')
plt.title(f"SLIC Segments")

plt.show()